In [0]:

%run ../source_to_bronze/utils

In [0]:

df = spark.read.table("dim_employee")
display(df)

In [0]:
from pyspark.sql.functions import *

salary_df = df.groupBy("department_id") \
    .agg(sum("salary").alias("total_salary")) \
    .orderBy(desc("total_salary"))
display(salary_df)

In [0]:
emp_count_df = df.groupBy("department_id", "country_id").count()
display(emp_count_df)

In [0]:

dept_country_df = df.select("department_id", "country_id").distinct()
display(dept_country_df )     

In [0]:
avg_age_df = df.groupBy("department_id") \
    .agg(avg("age").alias("avg_age"))

In [0]:
display(avg_age_df)

In [0]:
salary_df = salary_df.withColumn("at_load_date", current_date())
count_df = emp_count_df.withColumn("at_load_date", current_date())
dept_country_df = dept_country_df.withColumn("at_load_date", current_date())
avg_age_df = avg_age_df.withColumn("at_load_date", current_date())


In [0]:
salary_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save("/Volumes/workspace/default/assignment/gold/employee_salary")

count_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save("/Volumes/workspace/default/assignment/gold/fact_employee_count")

dept_country_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save("/Volumes/workspace/default/assignment/gold/dim_department_country")

avg_age_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save("/Volumes/workspace/default/assignment/gold/fact_employee_avg_age")